# CPE106L-4 Lesson 4: User-Centered Design and System Models

**Section:** FOPI01  
**Course project:** ChallengeHub  
**Assessment connection:** Integrated Design Project Output 3  
**AI use:** AI-Integrated

**TEAM01**


## Connect Course Project
Use the persistent course-project folder at `/content/drive/MyDrive/ChallengeHub`. This cell reconnects a new Colab runtime without re-uploading the repository.

In [1]:
from pathlib import Path
import os, sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

candidates = []
configured = os.environ.get("CHALLENGEHUB_ROOT")
if configured:
    candidates.append(Path(configured).expanduser().resolve())
if IN_COLAB:
    candidates.extend([Path("/content/drive/MyDrive/ChallengeHub"), Path("/content/ChallengeHub")])
current = Path.cwd().resolve()
candidates.extend([current, current / "ChallengeHub", *current.parents])

PROJECT_ROOT = next((p for p in candidates if (p / "challengehub").is_dir() and (p / "data").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "ChallengeHub was not found. In Colab, place it in My Drive/ChallengeHub; locally, run from the ChallengeHub project folder or set CHALLENGEHUB_ROOT."
    )

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
print("ChallengeHub connected:", PROJECT_ROOT)


Mounted at /content/drive
ChallengeHub connected: /content/drive/MyDrive/ChallengeHub


## Inspect the current Record Activity baseline
Use the starter implementation as evidence. Do not replace `ChallengeManager` with a later architecture in this lesson.

In [2]:
from challengehub.manager import ChallengeManager
manager = ChallengeManager(DATA_DIR)
challenge = next(c for c in manager.challenges if c.challenge_id == "C001")
{
    "challenge_id": challenge.challenge_id,
    "name": challenge.name,
    "activity_type": challenge.activity_type,
    "goal_unit": challenge.goal_unit,
    "goal_target": challenge.goal_target,
    "start_date": str(challenge.start_date),
    "end_date": str(challenge.end_date),
}

{'challenge_id': 'C001',
 'name': 'Walk 20K',
 'activity_type': 'walk',
 'goal_unit': 'km',
 'goal_target': 20.0,
 'start_date': '2026-08-01',
 'end_date': '2026-08-31'}

## Act 1 - Proto-persona and context
Create a focused design hypothesis. Label each statement as a project fact, earlier-model evidence, or an assumption requiring validation.

In [3]:
proto_persona = {
    "label": "Mika",
    "goal": {
        "statement":"Record valid activity progress correctly and verify recalculated results without repeated data entry.",
        "label": "[Earlier-Model Evidence] (UC-05 Spec & FB-02)"
    },
    "context_assumptions": [
        {"text": "Uses smartphone outdoors or while commuting shortly after completing a workout", "label": "[Assumption Requiring Validation]"},
        {"text": "Subject to screen glare, physical movement, interruptions, and partial attention", "label": "[Assumption Requiring Validation]"},
        {"text": "May experience intermittent mobile cellular connectivity", "label": "[Assumption Requiring Validation]"}
    ],
    "project_facts": [
        {"text": "Target Challenge C001 Walk 20K (activity_type='walk', goal_unit='km', target=20.0)", "label": "[Project Fact]"},
        {"text": "FB-04 State Protection Rule: Invalid input raises ValueError with zero state change (Δrows = 0)", "label": "[Project Fact]"},
        {"text": "FB-07 Persistence: Valid submission appends record to activities.csv", "label": "[Project Fact]"}
    ],
    "design_implications": [
        {"text": "Concise mobile layout with clear hierarchy and large touch targets", "label": "[Assumption Requiring Validation]"},
        {"text": "Prominent display of challenge name ('Walk 20K') and required unit ('km') beside input field", "label": "[Project Fact]"},
        {"text": "Inline plain-language validation feedback for invalid inputs (amount<=0)", "label": "[Earlier-Model Evidence]"},
        {"text": "Input recovery: Retain valid pending field inputs upon error retry", "label": "[Earlier-Model Evidence]"}
    ]
}

task_risks = [
    {
        "step": 1,
        "task_action": "Locate Record Activity entry point",
        "risk": "User cannot find trigger button for target active challenge",
        "evidence_label": "[Earlier-Model Evidence]",
        "design_response": "Prominent 'Record Activity' action button on active challenge view (Requirement U1)"
    },
    {
        "step": 2,
        "task_action": "Review challenge context and unit",
        "risk": "User misinterprets unit of measurement (entering miles instead of km)",
        "evidence_label": "[Project Fact]",
        "design_response": "Display challenge name ('Walk 20K') and required unit ('km') on entry form (Requirement U2)"
    },
    {
        "step": 3,
        "task_action": "Enter parameters (amount, date, note)",
        "risk": "User mistypes negative amount or out-of-range date on touchscreen",
        "evidence_label": "[Project Fact]",
        "design_response": "Validate field values inline and render plain-language feedback (Requirement U3)"
    },
    {
        "step": 4,
        "task_action": "Submit activity entry",
        "risk": "Double-tapping or network latency creates submit status uncertainty",
        "evidence_label": "[Assumption Requiring Validation]",
        "design_response": "Disable duplicate submit clicks and show visible progress feedback (Requirement U4)"
    },
    {
        "step": 5,
        "task_action": "Correct errors if validation fails",
        "risk": "System wipes entire form, forcing re-entry of valid fields",
        "evidence_label": "[Earlier-Model Evidence]",
        "design_response": "Retain valid pending inputs in form during validation retry (Requirement U5)"
    },
    {
        "step": 6,
        "task_action": "Verify save confirmation and progress update",
        "risk": "Uncertainty whether data was persisted or progress total updated",
        "evidence_label": "[Project Fact]",
        "design_response": "Render confirmation view displaying saved values and updated ProgressSummary (Requirement U4)"
    }
]

# Output summary verification
(proto_persona["label"], len(task_risks))

('Mika', 6)

### 1A. Labeled Proto-Persona Design Hypothesis (Mika)

To establish a clear user-centered boundary for `UC-05: Record Activity`, Team 1 defines the Mika proto-persona hypothesis. Mika represents an active participant logging activity progress shortly after completing an outdoor physical or productivity task. Every design statement is explicitly labeled as a `[Project Fact]`, `[Earlier-Model Evidence]`, or `[Assumption Requiring Validation]` to maintain design integrity and prevent ungrounded claims.

| Proto-Persona Attribute | Design Hypothesis Statement (*Mika*) | Evidence Classification Label |
| :--- | :--- | :---: |
| Primary Goal | Record valid activity progress (distance, duration) quickly and verify the updated challenge completion total without repeating data entry. | `[Earlier-Model Evidence]`<br>(UC-05 Main Success Scenario & PO1 Backlog `FB-02`) |
| Core Workflow | Logs activity by submitting `participant_id`, `challenge_id`, `activity_type`, `amount`, `activity_date`, and optional `note`. | `[Project Fact]`<br>(Phase 1 Data Schema & `activities.csv` Dataset) |
| Context of Use | Operates a personal mobile smartphone outdoors (post-workout) or while commuting. | `[Assumption Requiring Validation]`<br>(Plausible Environmental Context) |
| Environmental Constraints | Subject to outdoor screen glare, physical movement, partial attention, and intermittent cellular connectivity. | `[Assumption Requiring Validation]`<br>(Plausible Field Constraint) |
| Unit Uncertainty | Risk of misinterpreting measurement units (e.g., entering `miles` instead of `km` for challenge `C001 Walk 20K`). | `[Project Fact]`<br>(Baseline Dataset: `C001` specifies `goal_unit='km'`)* |
| User Frustrations | Form inputs wiped or reset upon validation error, generic unhelpful error messages, and uncertainty over whether a save succeeded. | `[Earlier-Model Evidence]`<br> (PO2 Defect Analysis & Error Handling Rules `FB-04`) |
| State Protection Rule | Rejected submissions must raise an explicit `ValueError` and leave stored records and progress totals strictly unchanged. | `[Project Fact]`<br>(Requirement `FB-04` & PO2 Invariant Enforcement) |
| Success Postcondition | Submitted values and dynamically recalculated progress (`ProgressSummary`) are immediately visible on screen. | `[Earlier-Model Evidence]`<br> (PO2 Sequence Return Message & Requirements `FB-05` / `FB-07`) |

### 1B. Environmental Context & Design Risk Analysis


| Context Factor | Evidence Classification | Environmental Design Risk | System Design Mitigation Response |
| :--- | :---: | :--- | :--- |
| Outdoor Smartphone Use | `[Assumption Requiring Validation]` | Outdoor glare and small display touch targets lead to input mistyping. | Concise layout, high visual contrast, and large touch input targets (`Requirement U1`) |
| Measurement Unit Ambiguity | `[Project Fact]` | User inputs magnitude without knowing if `C001` expects `km`, `miles`, or `steps`. | Prominently display required `goal_unit` (`km`) beside the amount input field (`Requirement U2`) |
| Field Validation Failure | `[Earlier-Model Evidence]` | System rejects invalid amount (`-5`) or out-of-bounds date, causing user confusion. | Display inline, field-specific error feedback in plain language (`Requirement U3`). |
| Intermittent Cellular Latency | `[Assumption Requiring Validation]` | Delayed network response leads to double-tapping and submit status uncertainty. | Disable repeated taps and display visible status/processing feedback (`Requirement U4`) |
| Task Interruption Mid-Entry | `[Assumption Requiring Validation]` | User gets distracted mid-form; validation failure wipes valid entries. | Preserve validly entered pending inputs (`date`, `note`) upon form re-display (`Requirement U5`) |


### 1C. Step-by-Step Task Risk & Evidence Mapping (`UC-05`)

This analysis breaks down the 6-step operational task flow for `UC-05: Record Activity`, mapping every user action to its primary risk, evidence classification, and mitigation requirement:

| Step # | Task Action Step | Primary User Risk (Error / Interruption / Uncertainty) | Evidence Classification Label | System Mitigation Requirement |
| :---: | :--- | :--- | :---: | :--- |
| 1 | Locate Entry Point | Cannot find where to log progress for active challenge `C001`. | `[Earlier-Model Evidence]`<br>(UC-05 Step 1) | Direct "Record Activity" button on active challenge card (`U1`) |
| 2 | Review Context | Misinterprets measurement unit (`km` vs. `miles`) or active date bounds. | `[Project Fact]`<br>(C001 Baseline Spec) | Explicitly display `name` (`Walk 20K`) and `goal_unit` (`km`) (`U2`) |
| 3 | Enter Values | Mistypes negative amount (`-5`) or out-of-bounds date on mobile screen. | `[Project Fact]`<br>(Requirement FB-04) | Field validation with plain-language inline error feedback (`U3`). |
| 4 | Submit Entry | Double-tapping or network delay creates uncertainty if submission worked. | `[Assumption Requiring Validation]`<br>(Mobile Network Context) | Disable duplicate submit clicks & show progress feedback (`U4`) |
| 5 | Correct Errors | System resets form upon error, forcing user to retype valid fields. | `[Earlier-Model Evidence]`<br>(PO2 Exception Flow) | Preserve valid pending field inputs (`date`, `note`) on retry (`U5`). |
| 6 | Verify Result | User is uncertain if activity was appended to `activities.csv` | `[Project Fact]`<br>(Requirement FB-07) | Render confirmation view with saved values and progress total (`U4`) |

## Act 2 - Usability requirements
Write at least five observable requirements with supporting evidence, priority, and an evaluation method.

In [4]:
requirements = [
    {
        "id": "U1",
        "requirement": "The system shall provide a prominent 'Record Activity' action button directly on the active challenge card (C001 Walk 20K).",
        "evidence": "UC-05 Step 1 evidence; PO1 Backlog FB-03; Act 1 Task Risk Step 1",
        "priority": "Must Have",
        "evaluation": "Timed Navigation Walkthrough (Participant locates and triggers entry form in <= 3 seconds)"
    },
    {
        "id": "U2",
        "requirement": "The activity logging form shall explicitly display the challenge name ('Walk 20K'), activity type ('walk'), measurement unit ('km'), and active date bounds ('2026-08-01' to '2026-08-31').",
        "evidence": "C001 Baseline dataset (goal_unit='km'); NFB-02 usability; Act 1 Task Risk Step 2",
        "priority": "Must Have",
        "evaluation": "Visual Clarity Inspection (100% of participants identify 'km' unit prior to entering data)"
    },
    {
        "id": "U3",
        "requirement": "When invalid data is entered (amount <= 0 or out-of-bounds date), the system shall reject submission and display inline, plain-language error feedback.",
        "evidence": "FB-04 State protection rule; PO2 Exception Paths; Act 1 Task Risk Step 3",
        "priority": "Must Have",
        "evaluation": "Error Injection Testing (System highlights field and renders 'Amount must be positive' within < 1 second)"
    },
    {
        "id": "U4",
        "requirement": "Upon successful save, the system shall render a confirmation view displaying saved values alongside updated ProgressSummary (total km, % complete).",
        "evidence": "FB-02 Progress view; FB-05 Recalculation; FB-07 Persistence; Act 1 Task Risk Step 6",
        "priority": "Must Have",
        "evaluation": "Task Completion Verification (Participant states updated total km & percentage without leaving screen)"
    },
    {
        "id": "U5",
        "requirement": "If validation fails or submission is interrupted, the system shall retain all validly entered pending field values (date, note) upon form re-display.",
        "evidence": "FB-04 Error path preservation; Mobile interruption context; Act 1 Task Risk Step 5",
        "priority": "Should Have",
        "evaluation": "Interruption & Recovery Test (100% of valid pending fields preserved during retry)"
    }
] # id, requirement, evidence, priority, evaluation

# Output evaluatoin
requirements

[{'id': 'U1',
  'requirement': "The system shall provide a prominent 'Record Activity' action button directly on the active challenge card (C001 Walk 20K).",
  'evidence': 'UC-05 Step 1 evidence; PO1 Backlog FB-03; Act 1 Task Risk Step 1',
  'priority': 'Must Have',
  'evaluation': 'Timed Navigation Walkthrough (Participant locates and triggers entry form in <= 3 seconds)'},
 {'id': 'U2',
  'requirement': "The activity logging form shall explicitly display the challenge name ('Walk 20K'), activity type ('walk'), measurement unit ('km'), and active date bounds ('2026-08-01' to '2026-08-31').",
  'evidence': "C001 Baseline dataset (goal_unit='km'); NFB-02 usability; Act 1 Task Risk Step 2",
  'priority': 'Must Have',
  'evaluation': "Visual Clarity Inspection (100% of participants identify 'km' unit prior to entering data)"},
 {'id': 'U3',
  'requirement': 'When invalid data is entered (amount <= 0 or out-of-bounds date), the system shall reject submission and display inline, plain-langu

### 2A. Testable Usability Requirements Specification (`U1`-`U5`)

To ensure that the user interface for `UC-05: Record Activity` directly mitigates the environmental glare, unit ambiguity and form reset risks identified in Act 1, Team 1 defines five testable usability requirements:

| Requirement ID | Focus Area | Detailed Usability Requirement Statement | Supporting Evidence Basis | MosCoW Priority | Evaluation Method & Pass Criteria |
| :---: | :--- | :--- | :--- | :---: | :--- |
| `U1` | Discovery & Trigger Path | The system shall provide a prominent "Record Activity" action button directly on the active challenge dashboard (`C001 Walk 20K`). | `UC-05` Step 1 / `FB-03` / Act 1 Task Risk 1 | Must Have |Timed Navigation Walkthrough: Participant locates and triggers form in $\le 3$ seconds |
| `U2` | Context & Unit Visibility | The activity logging form shall explicitly display the challenge name (`Walk 20K`), required activity type (`walk`), measurement unit (`km`), and active date bounds (`2026-08-01` to `2026-08-31`) | `C001` Dataset (`goal_unit='km'`) / `NFB-02` / Act 1 Task Risk 2 | Must Have | Visual Clarity Inspection: 100% of participants identify expected unit (`km`) prior to entry. |
| `U3` | Field Validation & Feedback | When invalid data is entered (amount $\le 0$ or out-of-bounds date), the system shall reject submission and display inline, plain-language error feedback. | `FB-04` State Protection Rule / PO2 Exception Paths / Act 1 Task Risk 3 | Must Have | Error Injection Testing: Highlights invalid field & displays `"Amount must be positive"` in $< 1$s. |
| `U4` | State Confirmation & Progress | Upon successful save, the system shall render a clear confirmation view displaying saved values (`amount`, `date`) alongside updated participant `ProgressSummary` (`total km`,`% complete`) | `FB-02` / `FB-05` / `FB-07` / Act 1 Task Risk 6 | Must Have | Task Completion Verification: Participant states updated total (`km`) & % without leaving screen |
| `U5` | Input Recovery & Preservation | If validation fails or submission is interrupted, the system shall retain all validly entered pending field values (`date`, `note`) upon form re-display. | `FB-04` Retry Rule / Mobile Interruption Context / Act 1 Task Risk 5 | Should Have | Interruption & Recovery Test: 100% of valid pending fields preserved during retry. |

### 2B. Evaluation Protocol & Baseline Test Mapping(`C001 Walk 20K`)

1. Normal Execution Walkthrough (`U1`, `U2`, `U4`):
   * Test Procedure: Participant views active challenge `C001 Walk 20K`, clicks Record Activity, enters `amount = 3.5 km` on date `2026-08-15`, and clicks submit.
   * Pass Criteria: Form displays unit `km`; navigation take `<=3s` ; confirmation view displays saved `3.5 km` and updated progress `3.5 / 20.0 km (17.5%)`

2. Validation & Input Recovery Scenario (`U3`, `U5`):
   * Test Procedure: Participant enters invalid amount `-5km` with date `2026-08-15` and note `"Morning walk"`
   * Pass Criteria: System rejects save, highlights amount field with error `"Amount must be positive"`, and retains `2026-08-15` and `"Morning walk"` intact in the form


## Act 3 - Interface states and annotations
Describe the entry, form, validation/recovery, and confirmation states. The confirmation shows the saved ActivityRecord and current progress calculated after the save.

In [5]:
interface_states = {
    "entry": (
        "Active Challenge Dashboard View: Displays 'C001 Walk 20K' summary card "
        "(Target: 20.0 km, Active: 2026-08-01 to 2026-08-31) with a prominent, high-contrast "
        "'Record Activity' primary action button."
    ),
    "form": (
        "Activity Logging Form View: Explicitly displays header context ('Walk 20K | Activity: walk'), "
        "active date bounds ('2026-08-01 to 2026-08-31'), input fields for Amount (with explicit 'km' unit label), "
        "Activity Date (YYYY-MM-DD), and optional Note, plus 'Submit Activity' and 'Cancel' buttons."
    ),
    "validation_recovery": (
        "Validation & Error Recovery View: Triggered upon invalid submission (e.g., Amount = -5.0 km). "
        "Highlights the invalid Amount field in red, displays inline plain-language error notice "
        "('ValueError: Amount must be positive'), and preserves validly entered pending inputs "
        "(Date = 2026-08-15, Note = 'Morning walk') intact."
    ),
    "confirmation": (
        "Save Confirmation & Progress Summary View: Displayed upon successful persistence. Shows explicit "
        "saved ActivityRecord card (Record ID: ACT-104, Amount: 3.5 km, Date: 2026-08-15) alongside "
        "dynamically recalculated ProgressSummary metrics (Total Logged: 3.5 / 20.0 km, Completion: 17.5%, Status: ACTIVE)."
    )
}

annotations = [
    {
        "decision": "Provide direct 'Record Activity' button on active challenge card (Entry State)",
        "evidence": "UC-05 Step 1; Act 1 Task Risk Step 1",
        "requirement": "U1 (Discovery & Trigger Path)",
        "rationale": "Reduces navigation friction for outdoor mobile users, enabling form access in <= 3 seconds."
    },
    {
        "decision": "Prominently display challenge name ('Walk 20K') and unit ('km') beside amount field (Form State)",
        "evidence": "C001 baseline spec (goal_unit='km'); Act 1 Unit Uncertainty Risk",
        "requirement": "U2 (Context & Unit Visibility)",
        "rationale": "Prevents unit ambiguity (e.g., entering miles or steps instead of km) under mobile glare."
    },
    {
        "decision": "Render inline plain-language error feedback ('Amount must be positive') on validation failure (Validation State)",
        "evidence": "FB-04 State Protection Rule; PO2 Exception Flow",
        "requirement": "U3 (Field Validation & Feedback)",
        "rationale": "Informs user of exact failure reason instantly without vague error codes or page redirects."
    },
    {
        "decision": "Preserve valid pending field entries (Date, Note) during error recovery (Validation State)",
        "evidence": "FB-04 Retry Rule; Mobile interruption context",
        "requirement": "U5 (Input Recovery & Preservation)",
        "rationale": "Eliminates frustrating data loss, ensuring zero re-typing of valid fields upon validation retry."
    },
    {
        "decision": "Display saved ActivityRecord alongside updated ProgressSummary total & percentage (Confirmation State)",
        "evidence": "FB-02 / FB-05 / FB-07 success postcondition",
        "requirement": "U4 (State Confirmation & Progress)",
        "rationale": "Provides immediate visual feedback that data was saved to activities.csv and progress was recalculated."
    }
] # decision, evidence, requirement, rationale

# Output evaluation tuple
(interface_states, annotations)

({'entry': "Active Challenge Dashboard View: Displays 'C001 Walk 20K' summary card (Target: 20.0 km, Active: 2026-08-01 to 2026-08-31) with a prominent, high-contrast 'Record Activity' primary action button.",
  'form': "Activity Logging Form View: Explicitly displays header context ('Walk 20K | Activity: walk'), active date bounds ('2026-08-01 to 2026-08-31'), input fields for Amount (with explicit 'km' unit label), Activity Date (YYYY-MM-DD), and optional Note, plus 'Submit Activity' and 'Cancel' buttons.",
  'validation_recovery': "Validation & Error Recovery View: Triggered upon invalid submission (e.g., Amount = -5.0 km). Highlights the invalid Amount field in red, displays inline plain-language error notice ('ValueError: Amount must be positive'), and preserves validly entered pending inputs (Date = 2026-08-15, Note = 'Morning walk') intact.",
  'confirmation': 'Save Confirmation & Progress Summary View: Displayed upon successful persistence. Shows explicit saved ActivityRecord c

### 3A. Low-Fidelity Wireframe States Overview (`UC-05 Record Activity`)

To satisfy the testable usability requirements (`U1`–`U5`), Team 1 designed four low-fidelity mobile interface states centered on Challenge `C001 Walk 20K`:

1. Entry State (`U1`): Navigation dashboard displaying active challenge details and a prominent Record Activity action button.
2. Form State (`U2`): Primary data entry form explicitly stating challenge context, required activity type (`walk`), measurement unit (`km`), and input fields.
3. Validation & Recovery State (`U3`, `U5`): Inline error feedback highlighting the invalid field (`amount = -5.0 km`) while preserving valid pending entries (`date`, `note`) intact.
4. Confirmation State (`U4`): Save confirmation screen showing the persisted `ActivityRecord` alongside dynamically recalculated `ProgressSummary` metrics.


### 3B. Interface Design Annotations & Traceability Table

| Interface State | Design Decision & Annotation | Supporting Evidence | Traceability Requirement | Design Rationale & Risk Mitigation |
| :--- | :--- | :--- | :---: | :--- |
| 1. Entry State | Direct "Record Activity" action button placed directly on active challenge card. | `UC-05` Step 1; Act 1 Task Risk Step 1 | `U1` | Enables rapid discovery ($\le 3$s) for outdoor mobile users without deep navigation friction. |
| 2. Form State | Header displays `Walk 20K`, `walk`, and explicit `km` unit label beside amount input. | `C001` Spec (`goal_unit='km'`); Act 1 Unit Risk | `U2` | Prevents measurement unit ambiguity under outdoor screen glare. |
| 3. Validation state | Invalid Amount field highlighted red with inline feedback `"ValueError: Amount must be positive"` | Requirement `FB-04`; PO2 Exception Path | `U3` | Renders immediate plain-language feedback ($< 1$s) without generic modal popups. |
| 3. Recovery State | Retains valid pending fields (`date = 2026-08-15`, `note = Morning walk`) upon validation retry. | Requirement `FB-04`; Mobile Interruption Context | `U5` | Eliminates frustrating data loss, ensuring zero re-typing of valid inputs during error recovery. |
| 4. Confirmation State | Renders saved `ActivityRecord` details (Distinct Step 1: Persistence) alongside recalculated `ProgressSummary` (Distinct Step 2: Progress Calculation). | Requirements `FB-02`, `FB-05`, `FB-07` | `U4` | Provides immediate verification that disk save succeeded and progress was calculated from stored records as distinct sequential operations. |

## Act 4 - Provisional system class model
Start from the current classes: `ChallengeManager`, `Participant`, `Challenge`, `Enrollment`, and `ActivityRecord`. A `RecordActivityUI` may be proposed only as a clearly labeled boundary hypothesis.


In [6]:
current_classes = ["ChallengeManager", "Participant", "Challenge", "Enrollment", "ActivityRecord"]
proposed_classes = ["RecordActivityUI"]
rejected_candidates = [
    {
        "candidate": "SubmitButton",
        "decision": "Reject",
        "reason": "Windget element. Not an independent software responsibility."
    },
    {
        "candidate": "DateInputPicker",
        "decision": "Reject",
        "reason": "Input widget capturing ActivityRecord.amount; lacks distinct domain or coordination behavior."
    },
    {
        "candidate": "ValidationNotifier",
        "decision": "Reject",
        "reason": "Transient view state/alert component. Error message presentation belongs to RecordActivityUI, while domain validation rules remain in ChallengeManager."
    },
    {
        "candidate": "ProgressSummary",
        "decision": "Reject",
        "reason": "Derived calculation result (a dictionary projection) returned dynamically by ChallengeManager.progress(), not an independent persistent entity."
    },
    {
        "candidate": "ActivityStorageAdapter",
        "decision": "Reject",
        "reason": "Premature architectural layering (storage module utility). Implementation mechanics are already supported via challengehub/storage.py."
    }
]
operations = [
    {
        "class": "RecordActivityUI",
        "operation": "collectInput()",
        "type": "boundary",
        "evidence": "Form state (U1, U2): Captures participant input fields from the active challenge context."
    },
    {
        "class": "RecordActivityUI",
        "operation": "showFieldError(message: str)",
        "type": "boundary",
        "evidence": "Validation/recovery state (U3, U5): Displays field-specific error while preserving valid entries."
    },
    {
        "class": "RecordActivityUI",
        "operation": "showConfirmation(record: ActivityRecord, progress: dict)",
        "type": "boundary",
        "evidence": "Confirmation state (U4): Renders the persisted ActivityRecord alongside recalculated progress."
    },
    {
        "class": "ChallengeManager",
        "operation": "record_activity(participant_id, challenge_id, activity_type, amount, activity_date, note)",
        "type": "coordinator/controller",
        "evidence": "UC-05 main workflow: Validates constraints, coordinates domain objects, and persists record."
    },
    {
        "class": "ChallengeManager",
        "operation": "progress(participant_id, challenge_id)",
        "type": "coordinator/controller",
        "evidence": "Confirmation requirement: Computes progress directly from stored activity records."
    },
    {
        "class": "Challenge",
        "operation": "Attributes: goal_unit, goal_target, start_date, end_date",
        "type": "domain entity",
        "evidence": "Provides measurement unit and validation bounds to manager."
    },
    {
        "class": "ActivityRecord",
        "operation": "Attributes: record_id, participant_id, challenge_id, amount, activity_date, note",
        "type": "domain entity",
        "evidence": "Stores quantitative activity log evidence."
    }
]
traceability = [
    {
        "id": "U1",
        "interface_state": "Entry State",
        "behavior_evidence": "Use-case trigger / Step 1 task risk",
        "system_responsibility": "RecordActivityUI exposes entry hook on active challenge card to initiate logging."
    },
    {
        "id": "U2",
        "interface_state": "Form State",
        "behavior_evidence": "Form setup / Challenge context",
        "system_responsibility": "Challenge supplies goal_unit, activity_type, and active date bounds via ChallengeManager."
    },
    {
        "id": "U3",
        "interface_state": "Validation State",
        "behavior_evidence": "Alternative path (invalid value rejection)",
        "system_responsibility": "ChallengeManager raises ValueError; RecordActivityUI translates to inline field error."
    },
    {
        "id": "U4",
        "interface_state": "Confirmation State",
        "behavior_evidence": "Save-then-verify sequence path",
        "system_responsibility": "ChallengeManager saves record and computes progress(); RecordActivityUI displays saved values."
    },
    {
        "id": "U5",
        "interface_state": "Correction State",
        "behavior_evidence": "Retry path / interrupted session",
        "system_responsibility": "RecordActivityUI retains uncommitted form data so valid entries remain visible upon error."
    }
]
(current_classes, proposed_classes, rejected_candidates, operations, traceability)

(['ChallengeManager',
  'Participant',
  'Challenge',
  'Enrollment',
  'ActivityRecord'],
 ['RecordActivityUI'],
 [{'candidate': 'SubmitButton',
   'decision': 'Reject',
   'reason': 'Windget element. Not an independent software responsibility.'},
  {'candidate': 'DateInputPicker',
   'decision': 'Reject',
   'reason': 'Input widget capturing ActivityRecord.amount; lacks distinct domain or coordination behavior.'},
  {'candidate': 'ValidationNotifier',
   'decision': 'Reject',
   'reason': 'Transient view state/alert component. Error message presentation belongs to RecordActivityUI, while domain validation rules remain in ChallengeManager.'},
  {'candidate': 'ProgressSummary',
   'decision': 'Reject',
   'reason': 'Derived calculation result (a dictionary projection) returned dynamically by ChallengeManager.progress(), not an independent persistent entity.'},
  {'candidate': 'ActivityStorageAdapter',
   'decision': 'Reject',
   'reason': 'Premature architectural layering (storage modu

### 4A. Candidate Inventory Box and Provisional System Class
| Candidate | Status | Required Reasoning |
| --- | --- | --- |
| `ChallengeManager` | Current system class | Coordinates validation, persistence, and dynamic progress calculation. |
| `Participant`, `Challenge`, `Enrollment`, `ActivityRecord` | Current domain classes | Retained approved core application domain entities and relevant state. |
| `storage` | Current implementation module | Implementation file utility; excluded from the domain class hierarchy. |
| `RecordActivityUI` | Proposed boundary hypothesis | Justified from prototype responsibilities: unifies form input, inline validation messaging, and confirmation. |
| `SubmitButton` / `AmountField` / `DateInputPicker` / `ValidationNotifier` | Reject as classes | Treated as individual UI interface elements/widgets inside the RecordActivityUI boundary. |
| `ProgressSummary` | Reject as class | Transient derived calculation dictionary returned by `ChallengeManager.progress()`, not a persistent entity. |
| `ActivityStorageAdapter` | Reject as class | Unnecessary architectural abstraction; file I/O is already directly handled by `challengehub/storage.py`. |

### 4B. Responsibility & Evidence Notes Box
| Evidence | Current/proposed operation | Class |
| --- | --- | --- |
| Captures participant input fields from active context (U1, U2) | `collectInput()` | `RecordActivityUI` hypothesis |
| Displays field-specific error while preserving entries (U3, U5) | `showFieldError(message: str)` | `RecordActivityUI` hypothesis |
| Renders persisted record alongside recalculated progress (U4) | `showConfirmation(record: ActivityRecord, progress: dict)` | `RecordActivityUI` hypothesis |
| Validates constraints, coordinates domain objects, persists record (UC-05) | `record_activity(...)` | `ChallengeManager` |
| Confirmation requirement: computes progress from activity records | `progress(...)` after save | `ChallengeManager` |
| Provides measurement unit and validation bounds to manager | show `goal_unit`, `goal_target`, `start_date`, `end_date` | `Challenge` through coordinator |
| Stores quantitative activity log evidence | Retain `record_id`, `amount`, `activity_date`, `note` attributes | `ActivityRecord` |

####4C. Cross-Artifact Traceability Matrix Box

| Requirement ID | Interface State | Behavior Evidence | System Responsibility |
| --- | --- | --- | --- |
| **U1** | Entry State | Use-case trigger (direct action on active card) | `RecordActivityUI` exposes navigation hook for `C001`. |
| **U2** | Form State | Challenge context verification | `Challenge.goal_unit` and boundaries supplied via `ChallengeManager`. |
| **U3** | Validation State | Alternative paths (exception on negative/out-of-bounds input) | `ChallengeManager` raises `ValueError`; `RecordActivityUI.showFieldError()` renders inline feedback. |
| **U4** | Confirmation State | Save-then-verify sequence path | `ChallengeManager.record_activity()` commits row; `ChallengeManager.progress()` computes updated totals; `RecordActivityUI.showConfirmation()` renders result. |
| **U5** | Recovery / Correction State | Retry path after validation rejection | `RecordActivityUI` buffers uncommitted form data so valid fields are preserved on re-render. |


## Act 5 - Project Output 3 release
Assign evidence to the notebook and draw.io pages, reconcile contradictions, and run the release gate.

In [8]:
release = {
    "notebook_evidence": [
        "Section 1: Ground in User Evidence (Mika Proto-Persona Hypothesis, Context Risk Matrix, Task Risk Mapping for UC-05)",
        "Section 2: Specify Usability (Testable Usability Requirements U1-U5, MoSCoW Priorities, C001 Walk 20K Baseline Evaluation Protocol)",
        "Section 3: Prototype the Task (4 Low-Fidelity Mobile Wireframes: Entry, Form, Validation/Recovery, Confirmation; Interface Annotations)",
        "Section 4: Refine System Class Model («boundary hypothesis» RecordActivityUI, Distinct Steps Rule, Cross-Artifact Traceability Matrix)",
        "Section 5: Release Packaging & Final Release Gate (File Naming Contract, 5-Point Quality Gate Audit, Ethical AI Disclosure, Team Approval)"
    ],
    "drawio_pages": [
        "Page 1: Persona & Context (Proto-Persona Card, Context Risk Matrix, Step-by-Step Task Risk Flow Table)",
        "Page 2: Usability Requirements (Requirements U1-U5 Table, C001 Baseline Grounding, Evaluation Execution Paths)",
        "Page 3: Prototype States (4 Low-Fidelity Wireframe Screen Cards, Design Annotations & Traceability Table)",
        "Page 4: System Class Model (UML Class Diagram with «boundary hypothesis» RecordActivityUI & Cross-Artifact Traceability Matrix)",
        "Page 5: Release Gate (5-Point Audit Checklist, Submission File Contract, Signed Team Release Approval Box)"
    ],
    "checks": [
        {
            "gate_id": "GATE-1",
            "act": "Act 1: Ground in User Evidence",
            "check": "Mika proto-persona explicitly labeled as a design hypothesis; environmental context glare and step-by-step task risks mapped.",
            "status": "PASSED"
        },
        {
            "gate_id": "GATE-2",
            "act": "Act 2: Specify Usability",
            "check": "5 testable usability requirements (U1-U5) defined with observable pass/fail criteria, MoSCoW priorities, and evaluation protocols.",
            "status": "PASSED"
        },
        {
            "gate_id": "GATE-3",
            "act": "Act 3: Prototype the Task",
            "check": "4 low-fidelity wireframe states modeled; Confirmation view highlights saving ActivityRecord (Step 1) and calculating ProgressSummary (Step 2) as distinct sequential steps.",
            "status": "PASSED"
        },
        {
            "gate_id": "GATE-4",
            "act": "Act 4: Refine System Class Model",
            "check": "System class model refined with «boundary hypothesis» RecordActivityUI; widget bloat rejected; U1-U5 cross-artifact traceability matrix fully closed.",
            "status": "PASSED"
        },
        {
            "gate_id": "GATE-5",
            "act": "Act 5: Release Packaging",
            "check": "Submission contracts verified under exact Team01 naming rules; AI disclosure completed; all 4 team members signed off.",
            "status": "PASSED"
        }
    ],
    "files": [
        "Team01_ProjectOutput3_UserSystemDesign.ipynb",
        "Team01_ProjectOutput3_UserSystemDesign.drawio",
    ],
}

# Output evaluation
release

{'notebook_evidence': ['Section 1: Ground in User Evidence (Mika Proto-Persona Hypothesis, Context Risk Matrix, Task Risk Mapping for UC-05)',
  'Section 2: Specify Usability (Testable Usability Requirements U1-U5, MoSCoW Priorities, C001 Walk 20K Baseline Evaluation Protocol)',
  'Section 3: Prototype the Task (4 Low-Fidelity Mobile Wireframes: Entry, Form, Validation/Recovery, Confirmation; Interface Annotations)',
  'Section 4: Refine System Class Model («boundary hypothesis» RecordActivityUI, Distinct Steps Rule, Cross-Artifact Traceability Matrix)',
  'Section 5: Release Packaging & Final Release Gate (File Naming Contract, 5-Point Quality Gate Audit, Ethical AI Disclosure, Team Approval)'],
 'drawio_pages': ['Page 1: Persona & Context (Proto-Persona Card, Context Risk Matrix, Step-by-Step Task Risk Flow Table)',
  'Page 2: Usability Requirements (Requirements U1-U5 Table, C001 Baseline Grounding, Evaluation Execution Paths)',
  'Page 3: Prototype States (4 Low-Fidelity Wireframe 

### Ethical AI-Assisted Use Disclosure Statement

**1. Tool Utilization & Scope:**  
Team 1 utilized Gemini Notebook(powered by Google Gemini Engine) as an administrative, formatting, and consultative AI tool during the execution of Project Output3. The scope of AI assistance was strictly limited to:
* Structuring and formatting 5 testable usability requirements (`U1`–`U5`), evaluation walkthrough methods, and MoSCoW priorities.
* Assisting in structuring ASCII layouts for 4 low-fidelity mobile wireframe states (`Entry`, `Form`, `Validation`, `Confirmation`).
* Guiding candidate class classification (`RecordActivityUI` boundary hypothesis) and cross-artifact traceability matrix mapping (`U1`–`U5`).

**2. Human Verification & Accountability Statement:**  
All design artifacts, interface wireframes, usability specifications, system class models, and traceability mappings generated with AI assistance were rigorously peer-reviewed and verified by human team members. The team cross-checked all AI-suggested metrics against actual project baseline data (`C001 Walk 20K`, `goal_unit='km'`), software invariant rules (`FB-04`), and `pytest` assertions. All final engineering decisions, architectural boundaries, and project deliverables remain the sole intellectual responsibility of Team 1.

**3. Team Sign-off**

| Member Name | Team Role & Responsibility Allocation | Approval Date | Release Status |
| :--- | :--- | :---: | :---: |
| **Rachel Joy Baldo** | **Project Manager & Requirements Lead** | September 13, 2026 | **Signed & Approved** |
| **Mark Angelo Aycardo** | **Software Architect & Lead Developer** | September 13, 2026 | **Signed & Approved** |
| **Jian Bernard Quinton** | **Git Integration Manager & Repository Lead** | September 13, 2026 | **Signed & Approved** |
| **Issa Dumlao** | **QA & Validation Lead** | September 13, 2026 | **Signed & Approved** |